# Multi-Agent Logistics — Demo

Visualize the PettingZoo `ParallelEnv`, run a short random rollout, and (optionally) load a trained PPO policy from `results/models/ppo_logistics.zip` after running `agents/train_marl.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from envs.logistics_multi_env import parallel_env

In [ ]:
env = parallel_env(num_agents=3, render_mode="rgb_array", grid_size=10)
obs, info = env.reset(seed=42)
frame = env.render()
plt.figure(figsize=(5, 5))
plt.imshow(frame)
plt.axis("off")
plt.title("Initial state (pickups=green, drops=blue, agents=colored disks)")
plt.show()

In [ ]:
returns = []
for ep in range(5):
    obs, _ = env.reset(seed=ep)
    total = 0.0
    while True:
        actions = {a: env.action_space(a).sample() for a in env.agents}
        obs, rew, term, trunc, _ = env.step(actions)
        total += sum(rew.values())
        if any(term.values()) or any(trunc.values()):
            break
    returns.append(total)

print("Random policy episode returns:", [round(r, 2) for r in returns])
print("Mean:", float(np.mean(returns)))

In [ ]:
# Optional: load PPO after training
from pathlib import Path

model_path = ROOT / "results" / "models" / "ppo_logistics.zip"
if model_path.is_file():
    from stable_baselines3 import PPO

    model = PPO.load(str(model_path))
    obs, _ = env.reset(seed=0)
    for _ in range(50):
        actions = {a: int(model.predict(obs[a], deterministic=True)[0]) for a in env.agents}
        obs, _, term, trunc, _ = env.step(actions)
        if any(term.values()) or any(trunc.values()):
            break
    plt.imshow(env.render())
    plt.axis("off")
    plt.title("After PPO rollout (partial)")
    plt.show()
else:
    print("Train first: python agents/train_marl.py")